# 1. Topologie et espaces normés

**Objectif** : illustrer numériquement les notions de complétude, compacité et convergence
dans différents espaces normés, et visualiser le théorème de Baire sur un exemple.

## Plan
1. Normes sur R^n : comparaison des boules unité
2. Suites de Cauchy et complétude : Q n'est pas complet, R l'est
3. Compacité : critère de Bolzano-Weierstrass en dimension finie
4. L'espace C([0,1]) muni de la norme sup est un espace de Banach
5. L'espace C([0,1]) muni de la norme L2 n'est pas complet : contre-exemple
6. Théorème de Baire : illustration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── 1. Boules unité pour les normes p = 1, 2, inf en dimension 2 ──────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
theta = np.linspace(0, 2 * np.pi, 1000)

for ax, p, title in zip(axes, [1, 2, np.inf], [r'$\ell^1$', r'$\ell^2$', r'$\ell^\infty$']):
    # Paramétrisation de la boule unité ||x||_p = 1
    if p == np.inf:
        x = np.array([-1, 1, 1, -1, -1])
        y = np.array([-1, -1, 1, 1, -1])
    else:
        # |cos(t)|^p + |sin(t)|^p = 1 => r(t) = 1 / (|cos t|^p + |sin t|^p)^{1/p}
        r = 1.0 / (np.abs(np.cos(theta))**p + np.abs(np.sin(theta))**p)**(1/p)
        x = r * np.cos(theta)
        y = r * np.sin(theta)
    ax.plot(x, y, 'steelblue', lw=2)
    ax.set_aspect('equal')
    ax.axhline(0, color='k', lw=0.5)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'Boule unité {title}', fontsize=12)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)

plt.suptitle('Boules unité en dimension 2', fontsize=14)
plt.tight_layout()
plt.savefig('../figures/boules_unit.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2. Suite de Cauchy dans Q qui ne converge pas dans Q ──────────────────────
# On approche sqrt(2) par la méthode de Newton : x_{n+1} = (x_n + 2/x_n) / 2
# La suite est de Cauchy mais sa limite sqrt(2) est irrationnelle.

def newton_sqrt2(n_iter=15):
    from fractions import Fraction
    x = Fraction(1)          # x_0 = 1 (rationnel)
    history = [x]
    for _ in range(n_iter):
        x = (x + Fraction(2, 1) / x) / 2
        history.append(x)
    return history

history = newton_sqrt2(12)
errors  = [abs(float(x) - np.sqrt(2)) for x in history]

plt.figure(figsize=(7, 4))
plt.semilogy(errors, 'o-', color='steelblue')
plt.xlabel('Itération n')
plt.ylabel('|x_n - √2|  (échelle log)')
plt.title('Convergence quadratique de Newton vers √2 dans R\n(suite rationnelle sans limite dans Q)')
plt.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.show()

print(f'x_12 = {history[12]}  ≈  {float(history[12]):.16f}')
print(f'√2   ≈  {np.sqrt(2):.16f}')
print(f'La suite est rationnelle à chaque étape mais converge vers un irrationnel.')

In [ ]:
# ── 3. C([0,1]) avec norme L2 n'est pas complet : suite de Cauchy sans limite continue ──
#
# On construit la suite de fonctions continues f_n convergeant dans L2 vers 1_{[1/2,1]}
# (indicatrice de [1/2, 1]), qui est discontinue et n'appartient pas à C([0,1]).
#
# f_n(x) =
#   0            si  x <= 1/2 - 1/n
#   n*(x-1/2)+1  si  1/2 - 1/n < x < 1/2
#   1            si  x >= 1/2

def f_n(x, n):
    out = np.zeros_like(x, dtype=float)
    mask_ramp = (x > 0.5 - 1/n) & (x < 0.5)
    out[mask_ramp] = n * (x[mask_ramp] - 0.5) + 1
    out[x >= 0.5] = 1.0
    return out

x = np.linspace(0, 1, 2000)
fig, ax = plt.subplots(figsize=(8, 4))
for n in [2, 5, 10, 30]:
    ax.plot(x, f_n(x, n), label=f'n = {n}')
ax.step([0, 0.5, 0.5, 1], [0, 0, 1, 1], 'k--', lw=2, label=r'$\mathbf{1}_{[1/2,1]}$ (limite L²)')
ax.set_title(r'Suite de Cauchy dans $(C([0,1]), \|\cdot\|_{L^2})$ sans limite continue', fontsize=12)
ax.set_xlabel('x')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Calcul numérique de ||f_n - f_m||_{L2}
print('Vérification que (f_n) est de Cauchy dans L2 :')
for n, m in [(5,10), (10,30), (30,100)]:
    diff_sq = np.trapezoid((f_n(x,n) - f_n(x,m))**2, x)
    print(f'  ||f_{n} - f_{m}||_L2 = {np.sqrt(diff_sq):.6f}')

## Conclusion

- $(C([0,1]), \|\cdot\|_{\sup})$ est un **espace de Banach** : toute suite de Cauchy converge dans l'espace.
- $(C([0,1]), \|\cdot\|_{L^2})$ **n'est pas complet** : la suite $(f_n)$ est de Cauchy pour $\|\cdot\|_{L^2}$
  mais sa limite $\mathbf{1}_{[1/2,1]}$ n'est pas continue. Le complété de cet espace est $L^2([0,1])$.
- Ce passage de $C([0,1])$ à $L^2([0,1])$ motive l'introduction des **espaces de Sobolev**
  comme cadre naturel pour les EDP (voir notebook 3).